In [2]:
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import os
import pandas as pd
import json

from qna_extractor import qna_extractor

2025-10-21 11:17:43 - gpd_ai_companion_qna_batch - INFO - Debug mode is disabled
2025-10-21 11:17:43 - gpd_ai_companion_qna_batch - INFO - Logger configured. Log files will be saved in: /Users/ananda_ganesan@uhc.com/Library/CloudStorage/OneDrive-UHG/Documents/GAK/gpd-ai-companion-transcripts-qna/logs
2025-10-21 11:17:43 - gpd_ai_companion_qna_batch - INFO - All required environment variables loaded successfully.


In [5]:
load_dotenv()

True

In [4]:
# Maximize display width for dataframes
pd.set_option('display.max_colwidth', None)  # Show full text in cells
pd.set_option('display.width', None)         # Use maximum display width
#pd.set_option('display.max_rows', None)      # Show all rows
#pd.set_option('display.max_columns', None)   # Show all columns

In [7]:
mira_search_client = SearchClient(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT_EASTUS"),"transcripts-mira",AzureKeyCredential(os.getenv("AZURE_SEARCH_API_KEY_EASTUS")))
qna_search_client = SearchClient(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT_EASTUS"),"transcripts-ucid-qna-v5",AzureKeyCredential(os.getenv("AZURE_SEARCH_API_KEY_EASTUS")))

Get All Negative questions and extract sentiment for conversation.

In [8]:
index_name = "transcripts-ucid-qna-v5"
select_fields = ["Ucid","sentiment_level_1"]
filter = f"StartTime ge 2025-10-20T00:00:00Z and StartTime lt 2025-10-21T00:00:00Z and sentiment_level_1 eq 'Negative'"
results = qna_search_client.search(select=select_fields,filter=filter)
df = pd.DataFrame(results, columns=select_fields)  
df.shape

(9580, 2)

In [9]:
counts = df.groupby('Ucid').size()
filtered_counts = counts[counts > 1].sort_values(ascending=False)
filtered_counts.shape

(1818,)

In [11]:
filtered_counts.head(10)

Ucid
31015008480562574882    7
00102074481760989425    6
00066553761761002716    6
00066414931760970090    6
00070391761760993042    6
00067276011760988462    6
00102128301760976397    6
00070541471761004269    6
00070339991760972868    6
34611392812181641856    6
dtype: int64

In [16]:
Ucids = ",".join(filtered_counts.index[:100])
print(Ucids)

31015008480562574882,00102074481760989425,00066553761761002716,00066414931760970090,00070391761760993042,00067276011760988462,00102128301760976397,00070541471761004269,00070339991760972868,34611392812181641856,00100336611760971001,00100394691761003182,00098005941760980330,00068576031760964165,00066077881760965165,00068316901760970699,00098605061760976599,00067172191761002698,05776341692457408084,00100595691760997184,00068063261760990998,00068110561760970510,00327139211762037859,00100167921760991164,00068108891760966106,00066525891760970280,00100429731760966784,00067201001760972797,29032341982474972805,00098058561760985132,00067426211760962396,39015228730864551219,00066012181760970228,25655247232485467431,00067376851760968212,00068534001761001110,00067391111760995799,00066038171760983653,00098064391760996775,00070236141760996978,00099535841760971519,00098087821760970508,00068461191760988203,00068244391760974875,00067512421760973920,00068282841760968613,00098233151760993438,0006819305176

In [17]:
filter =f"search.in(Ucid,'{Ucids}',',')"
select=["Ucid","Text","StartTime"]
results = mira_search_client.search(select=select, filter=filter)
df = pd.DataFrame(results)  
df.shape

(100, 7)

Get Positive Reports from qna_v5 index report.

In [5]:
filter="sentiment_level_1 eq 'Positive'" 
select=["Ucid","question","sentiment_level_1","sentiment_level_2","StartTime"]
qna_search_client = SearchClient(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT_EASTUS"),"transcripts-ucid-qna-v5",AzureKeyCredential(os.getenv("AZURE_SEARCH_API_KEY_EASTUS")))
positive_qna = qna_search_client.search(select=select, filter=filter)
df = pd.DataFrame(positive_qna)  
df.shape

(257, 9)

In [6]:
df[select].head(10)

,Ucid,question,sentiment_level_1,sentiment_level_2,StartTime
0,12662215750637887574,Was there anything I could do to help you out here?,Positive,Grateful,2025-10-03T15:35:04.772Z
1,00066509581760460594,Are you able to help me explore the new plan that is beneficial to me?,Positive,Satisfied,2025-10-14T17:17:45.293Z
2,00099232791760542993,I would like for you to go with me with the new Medicare plan that y all have.,Positive,Satisfied,2025-10-15T15:47:58.297Z
3,00067205741760466963,Can I complete the survey to give the agent a good rating?,Positive,Grateful,2025-10-14T18:40:54.389Z
4,05910133321161914422,How does the new plan with better benefits sound?,Positive,Other,2025-10-15T19:32:58.681Z
5,00100118761759964463,What benefits did you like most about your plan? I didn’t have copayments for one.,Positive,Satisfied,2025-10-08T23:03:33.813Z
6,25894133530896537232,I got a letter saying that I would get everything and I m real happy with United Healthcare.,Positive,Satisfied,2025-10-09T18:50:22.038Z
7,00100306231759948627,I was talking to a gentleman about a problem I was having with EU card and I would like to do a survey on him because he was excellent. How can I do that?,Positive,Grateful,2025-10-08T18:40:48.77Z
8,00098518961760037633,"I have till [date] to decide which plan I m on, but I m extremely happy with your plan so if I don t say anything does it stay the same or do I have to tell you right now what I m?",Positive,Satisfied,2025-10-09T19:23:00.846Z
9,00099243951760130404,What are the key benefits of the 07 plan that I like?,Positive,Satisfied,2025-10-10T21:08:42.767Z


In [33]:
df[select].to_csv('positive_qna_v5_140.csv', index=False)

In [7]:
Ucids = ",".join(df['Ucid'])
print(Ucids)

12662215750637887574,00066509581760460594,00099232791760542993,00067205741760466963,05910133321161914422,00100118761759964463,25894133530896537232,00100306231759948627,00098518961760037633,00099243951760130404,00066202471760649804,00066416811759754707,00098280311759756927,00067182261760574460,00102632331760710299,00100078271759340467,00070538701759334415,00100117501759337627,01296100330573048640,00068169001713988969,00098394221759778591,00102635421759847202,00099144371760027406,00066536171760122701,00099605481760102725,00099398111759764236,00102548001760567042,00070193041760034218,00102320291760027125,00067141221760647025,00098617611759861017,17256011731199850885,00098191651759844978,13460095430563701000,00098152661760535660,00098628971760626516,00068640801760013649,00144355541759277442,00100154351759420561,00068267801759414424,00068565281759426172,00066070911759451136,00099069851760546047,00066027331760544394,16691392010018309205,00102424721760392256,00100120781760301028,0006606560175

In [10]:
print(filter)

search.in(Ucid,12662215750637887574,00066509581760460594,00099232791760542993,00067205741760466963,05910133321161914422,00100118761759964463,25894133530896537232,00100306231759948627,00098518961760037633,00099243951760130404,00066202471760649804,00066416811759754707,00098280311759756927,00067182261760574460,00102632331760710299,00100078271759340467,00070538701759334415,00100117501759337627,01296100330573048640,00068169001713988969,00098394221759778591,00102635421759847202,00099144371760027406,00066536171760122701,00099605481760102725,00099398111759764236,00102548001760567042,00070193041760034218,00102320291760027125,00067141221760647025,00098617611759861017,17256011731199850885,00098191651759844978,13460095430563701000,00098152661760535660,00098628971760626516,00068640801760013649,00144355541759277442,00100154351759420561,00068267801759414424,00068565281759426172,00066070911759451136,00099069851760546047,00066027331760544394,16691392010018309205,00102424721760392256,0010012078176030102

In [11]:
#filter="StartTime gt 2025-10-15T17:49:50Z and StartTime lt 2025-10-15T17:50:00Z" #10 Records
#filter="StartTime gt 2025-10-15T17:47:00Z and StartTime lt 2025-10-15T17:50:00Z" # 137 Records
filter =f"search.in(Ucid,'{Ucids}',',')"
select=["Ucid","Text","StartTime"]
results = mira_search_client.search(select=select, filter=filter)
df = pd.DataFrame(results)  
df.shape

(243, 7)

In [18]:
df[['Ucid','Text','StartTime']].head(10)

,Ucid,Text,StartTime
0,00067376851760968212,Agent: Thank you for calling. My name is Donal...,2025-10-20T13:52:38.633Z
1,00326217782248626259,customer: If you record your name and reason f...,2025-10-20T17:43:18.675Z
2,00100606731760974187,"Agent: Hi, thank you so much for calling Unite...",2025-10-20T15:31:53.927Z
3,00067161541760973802,Agent: Thank you for calling. My name is Keely...,2025-10-20T15:25:08.976Z
4,00067464691760968356,"Agent: Hello, I do apologize for that. Thank y...",2025-10-20T14:02:24.464Z
5,00098058561760985132,Agent: Thank you for calling. This is Amanda B...,2025-10-20T18:37:43.192Z
6,00099573841760976288,Agent: Thank you for calling. How may I help y...,2025-10-20T16:13:47.726Z
7,00100648881760987361,Agent: Thank you for calling. My name is Zach....,2025-10-20T19:13:31.265Z
8,00066414931760970090,Agent: Good morning. Thank you for calling. My...,2025-10-20T14:46:21.142Z
9,00067542281760977951,Agent: Thank you for calling. My name is Kevin...,2025-10-20T16:36:41.024Z


In [19]:
def extract_sentiments(row):
    extractor = qna_extractor(question=row['Text'], project="MIRA")
    try:

        sentiment_level_1_json = extractor.extract_sentiment_level_one()
        cleaned_sentiment_1 = sentiment_level_1_json.strip()
        cleaned_sentiment_1 = cleaned_sentiment_1.replace("```json", "").replace("```", "").strip()
        cleaned_sentiment_1 = cleaned_sentiment_1.lstrip('\n').rstrip('\n')
        sentiment_level_1_data = json.loads(cleaned_sentiment_1)
        sentiment_level_1_value = sentiment_level_1_data.get('sentiment_level_1')
        #expanded_qna_1["sentiment_level_1"] = sentiment_level_1_value
        row['sentiment_level_1'] = sentiment_level_1_value 
    except Exception as e:
            print(f"Error processing Ucid {row['Ucid']}: {e}")
            row['sentiment_level_1'] = None
            row['sentiment_level_2'] = None
            return row
    try:
        sentiment_level_2_json = extractor.extract_sentiment_level_two(sentiment_level_1_value)
        # Clean the response before parsing
        cleaned_sentiment_2 = sentiment_level_2_json.strip()
        cleaned_sentiment_2 = cleaned_sentiment_2.replace("```json", "").replace("```", "").strip()
        cleaned_sentiment_2 = cleaned_sentiment_2.lstrip('\n').rstrip('\n')
        
        sentiment_level_2_data = json.loads(cleaned_sentiment_2)
        #expanded_qna_1["sentiment_level_2"] = sentiment_level_2_data.get('sentiment_level_2')
        row['sentiment_level_2'] = sentiment_level_2_data.get('sentiment_level_2')
        return row 
    except Exception as e:
            print(f"Error processing Ucid {row['Ucid']}: {e}")
            row['sentiment_level_2'] = None
            return row

df = df.apply(extract_sentiments, axis=1)
df.shape

2025-10-21 11:27:43 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-21 11:27:44 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-21 11:27:45 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-21 11:27:46 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-21 11:27:47 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-21 11:27:48 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-21 11:27:49 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-21 11:27:49 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-21 11:27:51 - gp

(100, 9)

In [20]:
df.groupby('sentiment_level_1').size()

sentiment_level_1
Negative    91
Neutral      9
dtype: int64

In [21]:
df[['Ucid','Text','StartTime','sentiment_level_1','sentiment_level_2']].head(10)

,Ucid,Text,StartTime,sentiment_level_1,sentiment_level_2
0,00067376851760968212,Agent: Thank you for calling. My name is Donal...,2025-10-20T13:52:38.633Z,Negative,Frustrated
1,00326217782248626259,customer: If you record your name and reason f...,2025-10-20T17:43:18.675Z,Negative,Frustrated
2,00100606731760974187,"Agent: Hi, thank you so much for calling Unite...",2025-10-20T15:31:53.927Z,Neutral,Procedural
3,00067161541760973802,Agent: Thank you for calling. My name is Keely...,2025-10-20T15:25:08.976Z,Neutral,Procedural
4,00067464691760968356,"Agent: Hello, I do apologize for that. Thank y...",2025-10-20T14:02:24.464Z,Negative,Frustrated
5,00098058561760985132,Agent: Thank you for calling. This is Amanda B...,2025-10-20T18:37:43.192Z,Negative,Frustrated
6,00099573841760976288,Agent: Thank you for calling. How may I help y...,2025-10-20T16:13:47.726Z,Negative,Frustrated
7,00100648881760987361,Agent: Thank you for calling. My name is Zach....,2025-10-20T19:13:31.265Z,Neutral,Informational
8,00066414931760970090,Agent: Good morning. Thank you for calling. My...,2025-10-20T14:46:21.142Z,Negative,Frustrated
9,00067542281760977951,Agent: Thank you for calling. My name is Kevin...,2025-10-20T16:36:41.024Z,Negative,Frustrated


In [19]:
df.groupby('sentiment_level_1').size()

sentiment_level_1
Negative     42
Neutral     165
Positive     34
dtype: int64

In [29]:
df[df['sentiment_level_1']=='Positive']['Text']

10    Agent: Good *********. My name is Joshua. How can I help you *****? customer: Oh my God, I'm so happy you speak English. customer: I can't even. Agent: I'm glad, I'm glad that you can find out there. I, I, I don't speak like 100%, but I can understand it and, and speak it. OK. customer: You can't believe the phone calls I at. customer: Where I can? Agent: You've been through a lot of in Spanish or other languages. customer: Oh my God, I you know, but I apologize for being so forward. But thank you. Agent: No, no worries, we're here to help. OK, OK, so how can I help you *****? customer: I wanted to. customer: Well, I have a policy with United Healthcare. Agent: Mm-hmm. Agent: Yes. customer: And I think I have to renew it, don't I? It's a renewal period coming up. Agent: So yes, we can check on that. If you can renew the same plan that you're having for ********* for t************ or if there is a better plan and I can see that it benefits you more, I could let you know the new pl

In [22]:
df[['Ucid','Text','sentiment_level_1','sentiment_level_2']].to_csv('conversation_sentiments_100_Negatives.csv', index=False)

In [16]:

extractor = qna_extractor(project="MIRA")
df['sentiment_level_1'] = df.apply(lambda row: extractor.extract_sentiment_level_one(row['Text']), axis=1)
#df['sentiment_level_2'] = df.apply(lambda row: extractor.extract_sentiment_level_two(row['sentiment_level_1'], row['Text']), axis=1)
df.shape

TypeError: qna_extractor.extract_sentiment_level_one() takes 1 positional argument but 2 were given